In [ ]:
import requests
import pandas as pd
import time
import unicodedata
import re
import aiohttp
import asyncio
import ssl
import nest_asyncio
from tqdm.asyncio import tqdm_asyncio

## Extracting Bills

In [ ]:
generalCourtNumber=range(182, 195)  # representing 182 for 2001-2002, 183 for 2003-2004, ....., 194 for 2025-2026
res=[]

for court_no in generalCourtNumber:
    url=f'https://malegislature.gov/api/GeneralCourts/{court_no}/Documents'
    resp=requests.get(url)

    # Handling non-success responses 
    if resp.status_code == 404:
        print(f"No data found for General Court {court_no}.")
        continue
    elif resp.status_code == 400:
        print(f"Bad request for General Court {court_no}.")
        continue
    elif not resp.ok:
        print(f"Error {resp.status_code} for General Court {court_no}.")
        continue

    # Parsing JSON 
    try:
        data=resp.json()
    except ValueError:
        print(f"Invalid JSON for General Court {court_no}.")
        continue

    # Extracting relevant fields if data exists
    for i in data:
        sponsor=i.get('PrimarySponsor') or {}
        res.append({
            'GeneralCourtNumber': court_no,
            'BillNumber': i.get('BillNumber'),
            'DocketNumber': i.get('DocketNumber'),
            'PrimarySponsorId': sponsor.get('Id'),
            'PrimarySponsorName': sponsor.get('Name'),
            'Title': i.get('Title')
        })

# Convert to DataFrame once at the end
Bill_details=pd.DataFrame(res)
print(Bill_details.head())


> Unable to fetch data for 182

In [ ]:
output_file='MA_Bill_Details.csv'
Bill_details.to_csv(output_file, index=False, encoding='utf-8')

## Roll Calls- Senate

In [ ]:
# Extract roll calls for given session
roll_calls=[]
def get_rollcalls(generalCourtNumber, branch, rollCallNumber):
    url = f'https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Branches/{branch}/RollCalls/{rollCallNumber}'
    
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()  # will raise for 400, 404, etc.
        data = resp.json()
    except requests.exceptions.HTTPError as e:
        print(f'Skipping Court {generalCourtNumber}, Roll Call {rollCallNumber}: {e}')
        return None


    # Create a dataframe storing required roll call data
    res=[]
    for i in data.keys(): 
        votes = data.get(i, [])  
        if not votes:            
            continue
        if i not in [ 'Yeas','Nays','Absent']: 
            continue
        if len(data[i])==0: 
            continue
        x=0
        for j in data[i]:
            member_url= j['Details']
            res.append({
                        
                        'GeneralCourtNumber': generalCourtNumber,
                        'Branch': branch, 
                        'RollCallNumber': rollCallNumber,
                        'MemberCode':j['MemberCode'], 
                        'MemberURL':j['Details'], 
                        'Vote':i})
    return res

generalCourtNumber = range(182, 195)
branch = 'Senate'

for court_no in generalCourtNumber:
    rollCallNumber = range(1,500)
    consecutive_failures=0 # to track continuous failures of requests. Assuming the roll calls have ended/ does not exists, moving to the next general court.

    for roll_call in rollCallNumber:
        data = get_rollcalls(court_no, branch, roll_call)

        if data:
            roll_calls.extend(data)
            consecutive_failures = 0  # reset on success
            
        else:
            consecutive_failures += 1
        # Stop if 10 consecutive failures
        if consecutive_failures >= 10:
            print(f"Stopping General Court {court_no} after 20 consecutive failed requests.")
            break

        time.sleep(0.5)
senate_votes = pd.DataFrame(roll_calls)


In [ ]:
senate_votes= pd.DataFrame(roll_calls)

## Members data

In [ ]:
# Leadership

generalCourtNumber = range(182, 195)
branches = ['Senate','House']
leadership = []

for branch in branches:
    for court_no in generalCourtNumber:
        
        url_leadership = f"https://malegislature.gov/api/GeneralCourts/{court_no}/Branches/{branch}/Leadership"
        resp_lead = requests.get(url_leadership)

        if resp_lead.status_code != 200:
            continue
        try:
            leaders = resp_lead.json()
        except ValueError:
            continue

        for leader in leaders:
            member_code = leader.get("Member", {}).get('MemberCode')
            if member_code:
                leadership.append({
                    'GeneralCourtNumber': court_no, 
                    'Branch': branch,
                    'MemberCode': member_code,
                    'Position': leader.get('Position')        
                })


In [ ]:
ma_leaders= pd.DataFrame(leadership)
print(ma_leaders)
print(ma_leaders['GeneralCourtNumber'].value_counts())

In [ ]:
ma_leaders.to_csv('MA_legislature_leaders.csv', index=False, encoding='utf-8')

In [ ]:
# Member extraction for each genral court

generalCourtNumber = [194]
branches = ['Senate']
member_records = []

for branch in branches:
    for court_no in generalCourtNumber:
        #Get all members for the General Court
        url_members = f"https://malegislature.gov/api/GeneralCourts/{court_no}/LegislativeMembers"
        resp = requests.get(url_members)

        if resp.status_code != 200:
            print(f"No member data found for General Court {court_no} for {branch}")
            continue
        try:
            members_data = resp.json()
            print(members_data)
        except ValueError:
            print(f"Invalid JSON for General Court {court_no}")
            continue
        
        #Combine both member and leadership info
        for m in members_data:
            member_url = m.get('Details')
            member_code= m.get('MemberCode')

            resp_mem = requests.get(member_url)
            
            if resp_mem.status_code in (400, 404):
                member_records.append({
                    'GeneralCourtNumber': court_no,
                    'MemberCode': member_code,
                    'FullName': '',
                    'District': '',
                    'Branch': '',
                    'Party': '',
                    'Active': '',
                    'LeadershipPosition': ''
                })
                continue
            
            resp_mem.raise_for_status()
            mem_details= resp_mem.json()
            member_records.append({
                'GeneralCourtNumber': court_no,
                'MemberCode': member_code,
                'FullName': mem_details.get('Name'),
                'District': mem_details.get('District'),
                'Branch': mem_details.get('Branch'),
                'Party': mem_details.get('Party'),
                'Active': mem_details.get('IsActive'),
                'LeadershipPosition': mem_details.get('LeadershipPosition')
            })
            time.sleep(0.5)
            print(mem_details)

members_df = pd.DataFrame(member_records)
# members_df.to_csv("Legislature_membership.csv", index=False, encoding='utf-8')


In [4]:
leg_mem_details= pd.read_csv('MA_legislature_members.csv')

cols_to_check = ['FullName', 'District', 'Branch', 'Party']
inconsistent_members = (
    leg_mem_details
    .groupby('MemberCode')[cols_to_check]
    .nunique()  
    .reset_index()
)
inconsistent_members = inconsistent_members[
    inconsistent_members[cols_to_check].gt(1).any(axis=1)
]
print(inconsistent_members)

    MemberCode  FullName  District  Branch  Party
16        AMG0         1         2       1      1
21        AMS2         2         1       1      1
25        AST1         2         1       1      1
27        AXP1         2         0       1      1
29        A_G0         2         1       1      1
73        CAR1         2         1       1      1
88        CMS1         3         1       1      1
94        CSC0         1         2       1      1
122       DFH1         4         1       1      1
149       E P1         2         0       1      1
183       GNP1         3         1       1      1
187       G_C1         2         1       1      1
203       JAH0         4         1       1      1
217       JCV0         1         2       1      1
220       JDP1         2         1       1      1
250       JML1         2         1       1      1
292       KES0         1         2       1      1
320       LDF1         2         1       1      1
325       LME0         1         2       1      1


In [5]:
dup_names= inconsistent_members[inconsistent_members['FullName'].gt(1)]
for i in dup_names.values:
    print(leg_mem_details[leg_mem_details['MemberCode']==i[0]][['MemberCode', 'FullName']])

    MemberCode                    FullName
50        AMS2  Alyson M. Sullivan-Almeida
232       AMS2  Alyson M. Sullivan-Almeida
430       AMS2          Alyson M. Sullivan
858       AMS2  Alyson M. Sullivan-Almeida
     MemberCode          FullName
1910       AST1  A. Stephen Tobin
1936       AST1       A. S. Tobin
2266       AST1       A. S. Tobin
2382       AST1       A. S. Tobin
2652       AST1       A. S. Tobin
     MemberCode                FullName
1938       AXP1  Anthony W. Petruccelli
2267       AXP1     Anthony Petruccelli
2383       AXP1     Anthony Petruccelli
2653       AXP1     Anthony Petruccelli
    MemberCode    FullName
194       A_G0  Adam Gómez
400       A_G0  Adam Gomez
424       A_G0  Adam Gomez
     MemberCode                  FullName
1383       CAR1  Cheryl A. Coakley-Rivera
1538       CAR1  Cheryl A. Coakley-Rivera
1756       CAR1  Cheryl A. Coakley-Rivera
2284       CAR1  Cheryl A. Coakley-Rivera
2357       CAR1          Cheryl A. Rivera
2673       CAR1      

In [6]:
def clean_text(s):
    if pd.isna(s):
        return s
    # accent
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    # punctuation
    s = re.sub(r'[^\w\s]', '', s)
    # spaces and lowercase
    s=re.sub(r'\s+', ' ', s).strip().lower()
    return s

def text_normalize(df):
    cols_to_check = ['FullName', 'District', 'Branch', 'Party']
    inconsistent_members = (
        df
        .groupby('MemberCode')[cols_to_check]
        .nunique()  
        .reset_index()
    )
    inconsistent_members = inconsistent_members[
        inconsistent_members[cols_to_check].gt(1).any(axis=1)
    ]
    # print(inconsistent_members)

    if not inconsistent_members.empty:
        df['FullName_clean'] = df['FullName'].apply(clean_text)
        df['District_clean'] = df['District'].apply(clean_text)

        # Keep record with the longest district after cleaning per MemberCode
        df['District_length'] = df['District_clean'].fillna('').apply(len)
        df = df.sort_values(['MemberCode', 'District_length'], ascending=[True, False])
        members_unique = df.drop_duplicates(subset=['MemberCode'], keep='first')
        members_unique = members_unique.drop(columns=['FullName_clean', 'District_clean', 'District_length'])
        
        # print('After cleaning: ')
        # inconsistent_members = (
        #     members_unique
        #     .groupby('MemberCode')[cols_to_check]
        #     .nunique()  
        #     .reset_index()
        # )
        # inconsistent_members = inconsistent_members[
        #     inconsistent_members[cols_to_check].gt(1).any(axis=1)
        # ]
        # print(inconsistent_members)

        return members_unique
    
    return df 


In [ ]:
# Member extraction for each general court

generalCourtNumber = range(182,194)
branches = ['Senate']
member_records=[]

for branch in branches:
    
    for court_no in generalCourtNumber:
        #Get all members for the General Court
        url_members = f"https://malegislature.gov/api/GeneralCourts/{court_no}/LegislativeMembers"
        resp = requests.get(url_members)

        if resp.status_code != 200:
            print(f"No member data found for General Court {court_no} for {branch}")
            continue
        try:
            members_data = resp.json()
            print(members_data)
        except ValueError:
            print(f"Invalid JSON for General Court {court_no}")
            continue
        
        leg_mem_details= text_normalize(leg_mem_details)


        #Combine both member and leadership info
        for m in members_data:
            member_url = m.get('Details')
            member_code= m.get('MemberCode')

            # -----------------------------------------------
            # Check if member already exists in extracted data
            # -----------------------------------------------
            if member_code in leg_mem_details['MemberCode'].values:
                existing = leg_mem_details.loc[
                    leg_mem_details['MemberCode'] == member_code
                ].iloc[0]

                # Get leadership position (if any)
                leader_match = leg_leaders[
                    (leg_leaders['GeneralCourtNumber'] == court_no) &
                    (leg_leaders['Branch'].str.lower() == branch.lower()) &
                    (leg_leaders['MemberCode'] == member_code)
                ]

                leadership_pos = (
                    leader_match['Position'].iloc[0]
                    if not leader_match.empty
                    else existing.get('LeadershipPosition', '')
                )

                member_records.append({
                    'GeneralCourtNumber': court_no,
                    'MemberCode': member_code,
                    'FullName': existing.get('FullName', ''),
                    'District': existing.get('District', ''),
                    'Branch': existing.get('Branch', ''),
                    'Party': existing.get('Party', ''),
                    'Active': existing.get('Active', ''),
                    'LeadershipPosition': leadership_pos
                })
                continue

            # -----------------------------------------------
            # If not present, fetch details via API
            # -----------------------------------------------
            resp_mem = requests.get(member_url)
            
            if resp_mem.status_code in (400, 404):
                member_records.append({
                    'GeneralCourtNumber': court_no,
                    'MemberCode': member_code,
                    'FullName': '',
                    'District': '',
                    'Branch': '',
                    'Party': '',
                    'Active': '',
                    'LeadershipPosition': ''
                })
                continue
            
            resp_mem.raise_for_status()
            mem_details= resp_mem.json()
            member_records.append({
                'GeneralCourtNumber': court_no,
                'MemberCode': member_code,
                'FullName': mem_details.get('Name'),
                'District': mem_details.get('District'),
                'Branch': mem_details.get('Branch'),
                'Party': mem_details.get('Party'),
                'Active': mem_details.get('IsActive'),
                'LeadershipPosition': mem_details.get('LeadershipPosition')
            })
            time.sleep(0.5)
            print(mem_details)

df = pd.DataFrame(member_records)
df.to_csv("MA_legislature_members.csv", index=False, encoding='utf-8')


In [14]:
bill_details= pd.read_csv('MA_Bill_Details.csv')

# Clean text fields
for col in ['BillNumber', 'DocketNumber', 'PrimarySponsorId', 'PrimarySponsorName', 'Title']:
    bill_details[col] = (
        bill_details[col]
        .astype(str)
        .str.strip()
        .replace({'nan': pd.NA})
    )

bill_details = bill_details.drop_duplicates()

bill_details = bill_details.dropna(subset=['BillNumber'], how='all')

print(bill_details.info())
bill_details.head()

<class 'pandas.core.frame.DataFrame'>
Index: 69214 entries, 9 to 80267
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   GeneralCourtNumber  69214 non-null  int64 
 1   BillNumber          69214 non-null  object
 2   DocketNumber        61392 non-null  object
 3   PrimarySponsorId    67566 non-null  object
 4   PrimarySponsorName  68618 non-null  object
 5   Title               69181 non-null  object
dtypes: int64(1), object(5)
memory usage: 3.7+ MB
None


,GeneralCourtNumber,BillNumber,DocketNumber,PrimarySponsorId,PrimarySponsorName,Title
9,184,H4176,HD94176,<NA>,<NA>,relative to HIV and Hepatitis C prevention
11,185,S2742,SD92742,<NA>,<NA>,An Act relative to landscape sprinkler systems
12,185,S2783,SD92783,S48,Senate Committee on Post Audit and Oversight,A report of the committee on Post Audit and Ov...
13,185,S2870,SD92870,S48,Senate Committee on Post Audit and Oversight,The Cost of Inaction: Climate Change in the Co...
14,186,S99,SD1,BET0,Bruce E. Tarr,An Act to provide for loans for waterfront dev...


## Bill-roll call mapping

In [ ]:
nest_asyncio.apply()

ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

async def fetch_rollcall(session, generalCourtNumber, billNumber, sem, retries=3):
    """Fetch roll call data for a given GeneralCourtNumber and BillNumber."""
    url = f"https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Documents/{billNumber}/RollCalls"

    async with sem:
        for attempt in range(1, retries + 1):
            try:
                async with session.get(url, ssl=ssl_context, timeout=15) as resp:
                    if resp.status in (404, 400):
                        return []  # no data found for this bill
                    resp.raise_for_status()
                    data = await resp.json()

                    # Extract roll call numbers
                    rows = []
                    for item in data:
                        roll_call = item.get("RollCallNumber")
                        if roll_call is not None:
                            rows.append({
                                "GeneralCourtNumber": generalCourtNumber,
                                "BillNumber": billNumber,
                                "RollCallNumber": roll_call
                            })
                    return rows

            except Exception as e:
                print(f" Error {generalCourtNumber}-{billNumber} attempt {attempt}: {e}")
                await asyncio.sleep(2 * attempt)

        return []


async def get_bill_rollcall_map(df, max_concurrent=50):
    sem = asyncio.Semaphore(max_concurrent)
    async with aiohttp.ClientSession() as session:
        tasks = [
            fetch_rollcall(session, row.GeneralCourtNumber, row.BillNumber, sem)
            for _, row in df.iterrows()
        ]
        results = await tqdm_asyncio.gather(*tasks)
    
    return [item for sublist in results for item in sublist]




In [ ]:
bill_details = pd.read_csv("MA_Bill_Details.csv")
bill_details = bill_details.dropna(subset=["BillNumber"])

results_flat = await get_bill_rollcall_map(bill_details, max_concurrent=50)
rollcall_df = pd.DataFrame(results_flat)
rollcall_df.to_csv("MA_Bill_rollcall_mapping.csv", index=False)